# GPU runs: EfficientNetV2-S and AST on the 40-species subset

The MLP, CNN and CRNN train on the team laptop (Apple M2). The two pretrained models
(EfficientNetV2-S, ~20 M parameters; AST, ~87 M) are too slow there, so they run here on a
free Colab / Kaggle T4 with **exactly the same code, split, cached inputs and metrics**
(`src/train.py`). Runs are listed in `experiments/species40_gpu.txt`.

Everything is resumable: features are cached on Google Drive after the first session, and the
queue skips runs whose result file already exists, so a disconnected session just continues.

Runtime > Change runtime type > **T4 GPU** before starting.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive (keeps the feature cache and results between sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/insect_dl'
!mkdir -p {DRIVE}/processed {DRIVE}/results

## 2. Code and dependencies

In [ ]:
%cd /content
!git clone -q https://github.com/ITWhizHarsh/insect-audio-dl.git || (cd insect-audio-dl && git pull -q)
%cd /content/insect-audio-dl
!pip install -q librosa soxr remotezip

## 3. Inputs for the 40-species subset

First session: fetch the 1,991 recordings listed in `data/meta/species40_manifest.csv` from
Zenodo by HTTP range requests (~1 GB, 20-40 min), build the cached log-mel / handcrafted
inputs, and copy the cache to Drive. Later sessions copy the cache back in under a minute.

In [ ]:
import os
CACHE = f'{DRIVE}/processed/species40'
if os.path.exists(f'{CACHE}/Test_logmel.npy'):
    !mkdir -p data/processed && cp -r {CACHE} data/processed/
else:
    !python -m src.data.fetch_subset --from-manifest data/meta/species40_manifest.csv --out data/raw/species40 --workers 12
    !python scripts/build_features.py --manifest data/raw/species40/manifest.csv --name species40 --workers 2
    !cp -r data/processed/species40 {DRIVE}/processed/
!ls -la data/processed/species40

## 4. Train

Each run prints one line per epoch and writes `results/metrics/<run>.json`,
`results/predictions/<run>_test.npz` and figures. Earlier results are copied back from Drive
first so finished runs are skipped.

In [ ]:
!mkdir -p results && cp -rn {DRIVE}/results/. results/ 2>/dev/null; true
!python scripts/run_queue.py experiments/species40_gpu.txt --dry-run

In [ ]:
!python scripts/run_queue.py experiments/species40_gpu.txt; cp -r results/. {DRIVE}/results/

## 5. Bring the results back to the repository

Download the zip (or take `insect_dl/results` from Drive), unzip it over the repository's
`results/` folder on the laptop, and commit the new files under the run owner's name.

In [ ]:
!tail -n 3 logs/*.log
!cd results && zip -qr /content/gpu_results.zip metrics predictions figures
from google.colab import files
files.download('/content/gpu_results.zip')